In [ ]:
import pandas as pd
import numpy as np
import os
import pyarrow.parquet as pq
import random

import data_preprocessing.kbase as kb

# Data Loading

In [ ]:
# Load master table with removes duplicates
master_table_raw = pq.read_table(os.path.join(kb.source_tables_path, "EDATOSCASO/master_table_raw.parquet")).to_pandas()



In [ ]:
master_table_raw.shape
master_table_raw.info()

# Main Vectors

In [ ]:
# Define the key and ID columns used in the function, if they aren't defined globally
key_col = 'DEMANDAPK'
id_columns = ['IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS']

In [ ]:
master_table_raw.head(5)

# Creation of PATIENTID variable

In [ ]:
import pandas as pd
import numpy as np


def create_hierarchical_patient_id(df, key_column, id_columns, birth_date_col='FNACIMIENTO'):
    """
    Creates a unique, random numeric PATIENTID for each patient group, 
    following strict hierarchical rules: P1 (IDTS) > P2 (DNI). 
    Records not linked by P1 or P2 (i.e., those with only NOMBRE_APELLIDOS 
    or having all IDs as None) will each receive a unique PATIENTID.

    Args:
        df (pd.DataFrame): The input DataFrame.
        key_column (str): The DEMANDAPK column (used as the base ID for unlinked records).
        id_columns (list): Identifier columns [IDTARJETASANITARIA, DNI, NOMBRE_APELLIDOS].
        birth_date_col (str): Birth date column (not used for linking).

    Returns:
        pd.DataFrame: The DataFrame with the new numeric 'PATIENTID' column.
    """
    
    df_result = df.copy()
    
    # Initialize temp_id with the original index. This serves as the unique ID for unlinked records.
    df_result['temp_id'] = np.arange(len(df_result))

    # --------------------------------------------------------------------------
    # 1. Apply Hierarchical Linking Logic (P1 and P2 ONLY)
    # --------------------------------------------------------------------------
    
    # --- PRIORITY 1: IDTARJETASANITARIA ---
    # Only link records that have a NON-NULL value in this column
    p1_col = id_columns[0]
    mask_p1_valid = pd.notna(df_result[p1_col])
    
    if mask_p1_valid.any():
        # Group by P1 and get minimum temp_id for each group
        p1_groups = df_result.loc[mask_p1_valid].groupby(p1_col)['temp_id'].transform('min')
        df_result.loc[mask_p1_valid, 'temp_id'] = p1_groups

    # --- PRIORITY 2: DNI ---
    # Only apply to records NOT linked by P1 and having NON-NULL DNI
    p2_col = id_columns[1]
    unlinked_mask = (df_result['temp_id'] == np.arange(len(df_result)))
    mask_p2_valid = unlinked_mask & pd.notna(df_result[p2_col])
    
    if mask_p2_valid.any():
        # Group by P2 and get minimum temp_id for each group
        p2_groups = df_result.loc[mask_p2_valid].groupby(p2_col)['temp_id'].transform('min')
        df_result.loc[mask_p2_valid, 'temp_id'] = p2_groups

    # --- PRIORITY 3: NOMBRE_APELLIDOS (NOT USED FOR LINKING) ---
    # Records still having temp_id == original index will remain as unique patients
    # This includes:
    # - Records with all IDs as None
    # - Records with only NOMBRE_APELLIDOS filled
    
    
    # --------------------------------------------------------------------------
    # 2. Unique Random Numeric ID Assignment
    # --------------------------------------------------------------------------
    
    # Get all unique patient groups
    unique_temp_ids = df_result['temp_id'].unique()
    
    # Generate unique random IDs (8 digits)
    min_range = 10_000_000
    max_range = 99_999_999
    
    # Safety check
    if len(unique_temp_ids) > (max_range - min_range + 1):
        raise ValueError("The number of unique patients exceeds the available random ID range.")
        
    # Generate unique random IDs
    random_ids = np.random.choice(
        range(min_range, max_range + 1), 
        size=len(unique_temp_ids), 
        replace=False
    )
    
    # Create mapping: temp_id -> random PATIENTID
    id_mapping = pd.Series(random_ids, index=unique_temp_ids)
    
    # Assign PATIENTID
    df_result['PATIENTID'] = df_result['temp_id'].map(id_mapping).astype('int64')

    # --------------------------------------------------------------------------
    # 3. Summary and Cleanup
    # --------------------------------------------------------------------------
    
    # Statistics
    final_count = df_result['PATIENTID'].nunique()
    total_records = len(df_result)
    
    # Count linking statistics
    p1_linked = (mask_p1_valid).sum() if 'mask_p1_valid' in locals() else 0
    p2_linked = (mask_p2_valid).sum() if 'mask_p2_valid' in locals() else 0
    unlinked = total_records - p1_linked - p2_linked
    
    print(f"\n{'='*60}")
    print(f"PATIENT ID ASSIGNMENT SUMMARY")
    print(f"{'='*60}")
    print(f"Total records processed: {total_records:,}")
    print(f"Total unique patients (PATIENTID): {final_count:,}")
    print(f"Average records per patient: {total_records / final_count:.2f}")
    print(f"\nLinking breakdown:")
    print(f"  - Linked by P1 (IDTARJETASANITARIA): {p1_linked:,} ({p1_linked/total_records*100:.1f}%)")
    print(f"  - Linked by P2 (DNI): {p2_linked:,} ({p2_linked/total_records*100:.1f}%)")
    print(f"  - Unlinked (unique IDs): {unlinked:,} ({unlinked/total_records*100:.1f}%)")
    print(f"{'='*60}\n")

    # Cleanup
    df_result = df_result.drop(columns=['temp_id'])
    
    return df_result

In [ ]:
# Call the hierarchical linking function
# The function will return a new DataFrame with the 'PATIENTID' column added.
master_table_raw_patientid = create_hierarchical_patient_id(
    df=master_table_raw,
    key_column=key_col,
    id_columns=id_columns
)

In [ ]:
selected_columns = ['DEMANDAPK'] + ['PATIENTID'] + ['IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS', 'FNACIMIENTO']
master_table_raw_patientid[selected_columns].head(100)

In [ ]:
# To view all rows where the PATIENTID value is repeated
# We use keep=False to mark ALL instances (the first and all subsequent duplicates) as True
duplicate_patient_id_rows = master_table_raw_patientid[
    master_table_raw_patientid['PATIENTID'].duplicated(keep=False)
].sort_values(by='PATIENTID')

# Display the duplicate rows and the relevant linking columns
print("Rows with Duplicate PATIENTIDs:")

In [ ]:
# Select the columns you want to view for verification
cols_to_view = ['DEMANDAPK', 'PATIENTID', 'IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS', 'FNACIMIENTO']
duplicate_patient_id_rows[cols_to_view].tail(100)

## Analysis of missing 

In [ ]:
# ============================================================================
# 1. CREATE SUBSET OF UNLINKED RECORDS
# ============================================================================

def create_unlinked_subset(df, id_columns):
    """
    Creates a subset with records that have:
    - All 3 ID variables as null, OR
    - Only NOMBRE_APELLIDOS not null (P1 and P2 are None)
    
    Args:
        df: Original DataFrame
        id_columns: List with [IDTARJETASANITARIA, DNI, NOMBRE_APELLIDOS]
    
    Returns:
        DataFrame with unlinked records
    """
    
    p1_col = id_columns[0]  # IDTARJETASANITARIA
    p2_col = id_columns[1]  # DNI
    p3_col = id_columns[2]  # NOMBRE_APELLIDOS
    
    # Condition 1: All 3 IDs are null
    all_null = (df[p1_col].isna() & df[p2_col].isna() & df[p3_col].isna())
    
    # Condition 2: Only NOMBRE_APELLIDOS is not null (P1 and P2 are null)
    only_name = (df[p1_col].isna() & df[p2_col].isna() & df[p3_col].notna())
    
    # Combine both conditions
    mask_unlinked = all_null | only_name
    
    df_unlinked = df[mask_unlinked].copy()
    
    # Summary
    print(f"\n{'='*70}")
    print(f"SUBSET: UNLINKED RECORDS (P1 and P2)")
    print(f"{'='*70}")
    print(f"Total records in original dataset: {len(df):,}")
    print(f"Unlinked records: {len(df_unlinked):,} ({len(df_unlinked)/len(df)*100:.2f}%)")
    print(f"\nBreakdown:")
    print(f"  - With all 3 IDs null: {all_null.sum():,}")
    print(f"  - Only with NOMBRE_APELLIDOS: {only_name.sum():,}")
    print(f"{'='*70}\n")
    
    return df_unlinked


# ============================================================================
# 2. MISSING VALUES ANALYSIS
# ============================================================================

def analyze_missing_values(df):
    """
    Analyzes NaN values and blank strings in the DataFrame
    
    Args:
        df: DataFrame to analyze
    
    Returns:
        Tuple with (missing_table, blank_table, combined_table)
    """
    
    print(f"\n{'='*70}")
    print(f"MISSING VALUES ANALYSIS")
    print(f"{'='*70}\n")
    
    # --- NaN values analysis ---
    missing_percentages = df.isnull().mean() * 100
    missing_table = missing_percentages.reset_index()
    missing_table.columns = ['Variable', 'Porcentaje_NaN']
    missing_table = missing_table.sort_values(by='Porcentaje_NaN', ascending=False)
    
    print("📊 NaN VALUES BY VARIABLE:")
    print("-" * 70)
    print(missing_table.to_string(index=False))
    print()
    
    # --- Blank strings analysis ---
    blank_percentages = (df.apply(lambda col: col.map(
        lambda x: isinstance(x, str) and x.strip() == ''
    )).mean() * 100)
    
    blank_table = blank_percentages.reset_index()
    blank_table.columns = ['Variable', 'Porcentaje_en_blanco']
    blank_table = blank_table.sort_values(by='Porcentaje_en_blanco', ascending=False)
    
    print("📊 BLANK VALUES (empty strings) BY VARIABLE:")
    print("-" * 70)
    print(blank_table.to_string(index=False))
    print()
    
    # --- Combined summary ---
    combined = missing_table.merge(
        blank_table, 
        on='Variable', 
        how='outer'
    )
    combined['Total_Faltante'] = combined['Porcentaje_NaN'] + combined['Porcentaje_en_blanco']
    combined = combined.sort_values(by='Total_Faltante', ascending=False)
    
    print("📊 COMBINED SUMMARY (NaN + Blank):")
    print("-" * 70)
    print(combined.to_string(index=False))
    print(f"\n{'='*70}\n")
    
    return missing_table, blank_table, combined


# ============================================================================
# 3. COMPLETE USAGE EXAMPLE
# ============================================================================

# Create subset of unlinked records
df_unlinked = create_unlinked_subset(df_final_patient_id, id_columns)

# Analyze missing values in the subset
missing_table, blank_table, combined_table = analyze_missing_values(df_unlinked)

# Optional: Save results
# df_unlinked.to_csv('unlinked_records.csv', index=False)
# combined_table.to_csv('missing_values_analysis.csv', index=False)

In [ ]:
df_unlinked.head(100)